# 05_attention_crowding · 注意力与交易拥挤

**核心经济逻辑**：成交异常集中、冲击成本高且价格路径噪声大时，交易更像拥挤或投机，短期存在反转；分散、低冲击的注意力更健康。

**预测周期**：1–3 个交易日

**关键构造**：crowding_score = z(amount_shock) + z(max_minute_share) + z(burstiness) + z(avg_trade_size_shock)

| 分量 | 权重 | 作用 |
|---|---|---|
| crowded_price_reversal = −ret_oc × positive(crowding) | +1.0 | 拥挤后的反转 |
| late_crowding_reversal = −ret_late30 × share_late30 × positive(crowding) | +0.8 | 尾盘拥挤修复 |
| max_minute_amount_share | −0.5 | 成交高度集中惩罚 |
| amount_absret_corr | −0.3 | 成交与波动共振惩罚 |
| crowding_score | −0.4 | 综合拥挤惩罚 |

**失效场景**：重大信息导致的集中成交可能延续；开盘/尾盘天然集中（已用 20 日冲击基准）；大单代理依赖分钟聚合。

In [ ]:
def main(datasources, start_date, end_date):
    """
    05_attention_crowding · 注意力与交易拥挤因子（自包含，可直接提交）。
    """
    import numpy as np
    import pandas as pd
    import dai

    if isinstance(datasources, dict):
        bar1m_table = (
            datasources.get("bar1m")
            or datasources.get("stock_bar1m")
            or datasources.get("bigalpha_2026_stock_bar1m")
        )
    else:
        bar1m_table = datasources
    if not bar1m_table:
        raise ValueError(f"未找到 bar1m 数据源，datasources={datasources}")

    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    ext_start = start_ts - pd.Timedelta(days=30)

    def fmt_ts(x):
        return pd.to_datetime(x).strftime("%Y-%m-%d %H:%M:%S")

    sql = f"""
WITH raw AS (
    SELECT
        date_trunc('day', date)::DATE AS trade_date,
        date AS bar_time,
        instrument,
        pre_close, open, high, low, close,
        deal_number, volume, amount,
        bid_price1, bid_price2, bid_price3, bid_price4, bid_price5,
        ask_price1, ask_price2, ask_price3, ask_price4, ask_price5,
        bid_volume1, bid_volume2, bid_volume3, bid_volume4, bid_volume5,
        ask_volume1, ask_volume2, ask_volume3, ask_volume4, ask_volume5,
        bid_num_orders1, bid_num_orders2, bid_num_orders3, bid_num_orders4, bid_num_orders5,
        ask_num_orders1, ask_num_orders2, ask_num_orders3, ask_num_orders4, ask_num_orders5
    FROM {bar1m_table}
    WHERE date BETWEEN TIMESTAMP '{fmt_ts(ext_start)}' AND TIMESTAMP '{fmt_ts(end_ts)}'
),
diff AS (
    SELECT
        trade_date, bar_time, instrument,
        open, high, low, close, pre_close,
        volume, amount, deal_number,
        bid_price1, bid_price2, bid_price3, bid_price4, bid_price5,
        ask_price1, ask_price2, ask_price3, ask_price4, ask_price5,
        bid_volume1, bid_volume2, bid_volume3, bid_volume4, bid_volume5,
        ask_volume1, ask_volume2, ask_volume3, ask_volume4, ask_volume5,
        bid_num_orders1, ask_num_orders1,
        GREATEST(volume - LAG(volume, 1, volume) OVER w, 0) AS vol_delta,
        GREATEST(amount - LAG(amount, 1, amount) OVER w, 0) AS amt_delta,
        GREATEST(deal_number - LAG(deal_number, 1, deal_number) OVER w, 0) AS trades_delta,
        close / NULLIF(LAG(close, 1, pre_close) OVER w, 0) - 1.0 AS r,
        CAST(bar_time AS TIME) AS tm,
        (bid_price1 + ask_price1) / 2.0 AS mid,
        (ask_price1 - bid_price1) / NULLIF((bid_price1 + ask_price1) / 2.0, 0) * 10000.0 AS spread_bps,
        bid_volume1 + bid_volume2 + bid_volume3 + bid_volume4 + bid_volume5 AS bid_depth5,
        ask_volume1 + ask_volume2 + ask_volume3 + ask_volume4 + ask_volume5 AS ask_depth5,
        bid_num_orders1 + bid_num_orders2 + bid_num_orders3 + bid_num_orders4 + bid_num_orders5 AS bid_orders5,
        ask_num_orders1 + ask_num_orders2 + ask_num_orders3 + ask_num_orders4 + ask_num_orders5 AS ask_orders5
    FROM raw
    WHERE bid_price1 > 0 AND ask_price1 > 0
    WINDOW w AS (PARTITION BY trade_date, instrument ORDER BY bar_time)
),
calc AS (
    SELECT
        trade_date, bar_time, instrument,
        open, high, low, close, pre_close,
        volume, amount, deal_number,
        vol_delta, amt_delta, trades_delta, r, tm, mid, spread_bps,
        bid_depth5, ask_depth5, bid_orders5, ask_orders5,
        bid_price1, ask_price1, bid_volume1, ask_volume1,
        bid_price2, ask_price2, bid_volume2, ask_volume2,
        bid_price3, ask_price3, bid_volume3, ask_volume3,
        bid_price4, ask_price4, bid_volume4, ask_volume4,
        bid_price5, ask_price5, bid_volume5, ask_volume5,
        bid_num_orders1, ask_num_orders1,
        mid - LAG(mid) OVER w AS dmid,
        LAG(bid_price1) OVER w AS lag_bid_p1,
        LAG(ask_price1) OVER w AS lag_ask_p1,
        LAG(bid_volume1) OVER w AS lag_bid_v1,
        LAG(ask_volume1) OVER w AS lag_ask_v1,
        (bid_depth5 + ask_depth5) AS depth5,
        (bid_depth5 + ask_depth5) - LAG(bid_depth5 + ask_depth5) OVER w AS ddepth5,
        LEAD(close, 10) OVER w AS close_fwd10,
        LEAD(spread_bps, 10) OVER w AS spread_fwd10,
        LEAD(bid_depth5 + ask_depth5, 10) OVER w AS depth5_fwd10,
        LEAD(r, 5) OVER w AS r_fwd5,
        LAG(r) OVER w AS r_lag,
        SUM(LN(GREATEST(1 + r, 1e-6))) OVER wr AS cumlog,
        SUM(amt_delta) OVER p AS amt_day,
        STDDEV(r) OVER p AS r_std_day,
        (bid_volume1 - ask_volume1) / NULLIF(bid_volume1 + ask_volume1, 0) AS obi1,
        (bid_depth5 - ask_depth5) / NULLIF(bid_depth5 + ask_depth5, 0) AS obi5,
        (bid_orders5 - ask_orders5) / NULLIF(bid_orders5 + ask_orders5, 0) AS order_imbalance5,
        (ask_price1 * bid_volume1 + bid_price1 * ask_volume1) / NULLIF(bid_volume1 + ask_volume1, 0)
            / NULLIF(mid, 0) - 1.0 AS micro_dev1,
        (ask_price1 * bid_volume1 + ask_price2 * bid_volume2 + ask_price3 * bid_volume3
         + ask_price4 * bid_volume4 + ask_price5 * bid_volume5
         + bid_price1 * ask_volume1 + bid_price2 * ask_volume2 + bid_price3 * ask_volume3
         + bid_price4 * ask_volume4 + bid_price5 * ask_volume5)
            / NULLIF(bid_depth5 + ask_depth5, 0) / NULLIF(mid, 0) - 1.0 AS vamp_dev5,
        (bid_volume1 + bid_volume2 - ask_volume1 - ask_volume2)
            / NULLIF(bid_volume1 + bid_volume2 + ask_volume1 + ask_volume2, 0) AS near_depth_asym,
        (bid_volume1 / NULLIF(bid_num_orders1, 0) - ask_volume1 / NULLIF(ask_num_orders1, 0))
            / NULLIF(bid_volume1 / NULLIF(bid_num_orders1, 0) + ask_volume1 / NULLIF(ask_num_orders1, 0), 0) AS avg_order_size_asym
    FROM diff
    WINDOW w AS (PARTITION BY trade_date, instrument ORDER BY bar_time),
           wr AS (PARTITION BY trade_date, instrument ORDER BY bar_time ROWS UNBOUNDED PRECEDING),
           p AS (PARTITION BY trade_date, instrument)
),
path AS (
    SELECT
        calc.*,
        MAX(cumlog) OVER wr AS peaklog,
        cumlog - MAX(cumlog) OVER wr AS dd,
        (CASE WHEN bid_price1 >= lag_bid_p1 THEN bid_volume1 ELSE 0 END
         - CASE WHEN bid_price1 <= lag_bid_p1 THEN lag_bid_v1 ELSE 0 END
         - CASE WHEN ask_price1 <= lag_ask_p1 THEN ask_volume1 ELSE 0 END
         + CASE WHEN ask_price1 >= lag_ask_p1 THEN lag_ask_v1 ELSE 0 END) AS ofi,
        SIGN(dmid) * amt_delta AS signed_amt
    FROM calc
    WINDOW wr AS (PARTITION BY trade_date, instrument ORDER BY bar_time ROWS UNBOUNDED PRECEDING)
)
SELECT
    trade_date::DATETIME AS date,
    instrument,
    ARG_MIN(open, bar_time) AS open,
    MAX(high) AS high,
    MIN(low) AS low,
    ARG_MAX(close, bar_time) AS close,
    ARG_MAX(pre_close, bar_time) AS pre_close,
    ARG_MAX(volume, bar_time) AS volume,
    ARG_MAX(amount, bar_time) AS amount,
    ARG_MAX(deal_number, bar_time) AS deal_number,
    ARG_MAX(amount, bar_time) / NULLIF(ARG_MAX(volume, bar_time), 0) AS vwap,
    ARG_MAX(close, bar_time) / NULLIF(ARG_MAX(pre_close, bar_time), 0) - 1.0 AS ret_cc,
    ARG_MAX(close, bar_time) / NULLIF(ARG_MIN(open, bar_time), 0) - 1.0 AS ret_oc,
    ARG_MAX(CASE WHEN tm <= TIME '10:00:00' THEN close END,
            CASE WHEN tm <= TIME '10:00:00' THEN bar_time END)
        / NULLIF(ARG_MIN(open, bar_time), 0) - 1.0 AS ret_open30,
    ARG_MAX(close, bar_time)
        / NULLIF(ARG_MIN(CASE WHEN tm >= TIME '14:30:00' THEN close END,
                         CASE WHEN tm >= TIME '14:30:00' THEN bar_time END), 0) - 1.0 AS ret_late30,
    SUM(CASE WHEN tm >= TIME '14:30:00' THEN amt_delta ELSE 0 END)
        / NULLIF(SUM(amt_delta), 0) AS amount_share_late30,
    SUM(CASE WHEN tm >= TIME '14:30:00' THEN r * r ELSE 0 END) AS rv_late30,
    SUM(CASE WHEN tm >= TIME '14:30:00' THEN r * r ELSE 0 END)
        / NULLIF(SUM(r * r), 0) AS late_rv_share,
    SUM(r * r) AS realized_var,
    SUM(CASE WHEN r < 0 THEN r * r ELSE 0 END) / NULLIF(SUM(r * r), 0) AS downside_share,
    MAX(ABS(r)) / NULLIF(STDDEV(r), 0) AS jump_ratio_proxy,
    CORR(r, r_lag) AS minute_ret_autocorr,
    CORR(amt_delta, ABS(r)) AS amount_absret_corr,
    MAX(amt_delta) / NULLIF(SUM(amt_delta), 0) AS max_minute_amount_share,
    -SUM(CASE WHEN amt_delta > 0 AND amt_day > 0
              THEN (amt_delta / amt_day) * LN(amt_delta / amt_day) ELSE 0 END)
        / NULLIF(LN(COUNT(*)), 0) AS amount_entropy,
    (STDDEV(amt_delta) - AVG(amt_delta))
        / NULLIF(STDDEV(amt_delta) + AVG(amt_delta), 0) AS amount_burstiness,
    ARG_MAX(amount, bar_time) / NULLIF(ARG_MAX(deal_number, bar_time), 0) AS average_trade_amount,
    ABS(ARG_MAX(close, bar_time) / NULLIF(ARG_MIN(open, bar_time), 0) - 1.0)
        / (SUM(ABS(r)) + 1e-12) AS path_efficiency,
    SUM(r * r) FILTER (tm >= TIME '09:30:00' AND tm < TIME '10:00:00') AS rv_b1,
    SUM(r * r) FILTER (tm >= TIME '10:00:00' AND tm < TIME '10:30:00') AS rv_b2,
    SUM(r * r) FILTER (tm >= TIME '10:30:00' AND tm < TIME '11:00:00') AS rv_b3,
    SUM(r * r) FILTER (tm >= TIME '11:00:00' AND tm < TIME '11:30:00') AS rv_b4,
    SUM(r * r) FILTER (tm >= TIME '13:00:00' AND tm < TIME '13:30:00') AS rv_b5,
    SUM(r * r) FILTER (tm >= TIME '13:30:00' AND tm < TIME '14:00:00') AS rv_b6,
    SUM(r * r) FILTER (tm >= TIME '14:00:00' AND tm < TIME '14:30:00') AS rv_b7,
    SUM(r * r) FILTER (tm >= TIME '14:30:00') AS rv_b8,
    MIN(dd) AS max_drawdown,
    ARG_MAX(dd, bar_time) AS end_drawdown,
    ARG_MAX(dd, bar_time) - MIN(dd) AS drawdown_recovery,
    AVG(CASE WHEN r < -2.0 * r_std_day THEN close_fwd10 / close - 1.0 END) AS price_recovery,
    AVG(spread_bps) AS spread_bps_mean,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN spread_bps END) AS spread_bps_tail,
    AVG(depth5) AS depth5_mean,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN depth5 END) AS depth5_tail,
    AVG(obi1) AS obi1_mean,
    AVG(obi5) AS obi5_mean,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN obi5 END) AS obi5_tail,
    AVG(order_imbalance5) AS order_imbalance5_mean,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN order_imbalance5 END) AS order_imbalance5_tail,
    AVG(micro_dev1) AS micro_dev1_mean,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN micro_dev1 END) AS micro_dev1_tail,
    AVG(vamp_dev5) AS vamp_dev5_mean,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN vamp_dev5 END) AS vamp_dev5_tail,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN near_depth_asym END) AS near_depth_asymmetry_tail,
    AVG(CASE WHEN tm >= TIME '14:30:00' THEN avg_order_size_asym END) AS avg_order_size_asymmetry_tail,
    AVG(CASE WHEN vol_delta = 0 AND dmid = 0 THEN 1.0 ELSE 0.0 END) AS quote_staleness,
    SUM(ofi) AS ofi_sum,
    SUM(CASE WHEN tm >= TIME '14:30:00' THEN ofi ELSE 0 END)
        / NULLIF(SUM(CASE WHEN tm >= TIME '14:30:00' THEN bid_volume1 + ask_volume1 ELSE 0 END), 0) AS ofi1_tail_norm,
    SUM(signed_amt) / NULLIF(SUM(amt_delta), 0) AS signed_amount_imbalance,
    SUM(CASE WHEN tm >= TIME '14:30:00' THEN signed_amt ELSE 0 END)
        / NULLIF(SUM(CASE WHEN tm >= TIME '14:30:00' THEN amt_delta ELSE 0 END), 0) AS signed_imbalance_tail,
    CORR(signed_amt, r) AS flow_price_alignment,
    CORR(ofi, r_fwd5) AS ofi_future5_corr,
    AVG(CASE WHEN r < -2.0 * r_std_day THEN (spread_bps - spread_fwd10) / NULLIF(spread_bps, 0) END) AS spread_recovery,
    AVG(CASE WHEN r < -2.0 * r_std_day THEN (depth5_fwd10 - depth5) / NULLIF(depth5, 0) END) AS depth_recovery,
    SUM(GREATEST(-ddepth5, 0) * (CASE WHEN vol_delta = 0 THEN 1.0 ELSE 0.0 END))
        / NULLIF(SUM(GREATEST(-ddepth5, 0)), 0) AS cancel_ratio,
    STDDEV(ddepth5) / NULLIF(AVG(depth5), 0) AS depth_change_vol
FROM path
GROUP BY trade_date, instrument
"""

    df = dai.query(sql, filters={"date": [ext_start, end_ts]}, compression=True).df()
    if df.empty:
        return pd.DataFrame(columns=["date", "instrument", "factor"])

    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["instrument"] = df["instrument"].astype(str)
    for col in [c for c in df.columns if c not in ("date", "instrument")]:
        df[col] = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)
    df = df.sort_values(["instrument", "date"]).reset_index(drop=True)

    def cs_zscore(s):
        s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
        valid = s.dropna()
        if len(valid) < 5:
            return pd.Series(np.nan, index=s.index)
        lo, hi = valid.quantile(0.01), valid.quantile(0.99)
        s2 = s.clip(lo, hi)
        std = s2.std(ddof=0)
        if pd.isna(std) or std < 1e-12:
            return pd.Series(0.0, index=s.index)
        return (s2 - s2.mean()) / std

    def z(d, col):
        return d.groupby("date")[col].transform(cs_zscore)

    def make_composite(data, out_col, components):
        num = pd.Series(0.0, index=data.index)
        den = pd.Series(0.0, index=data.index)
        for raw_col, weight in components:
            zc = z(data, raw_col)
            mask = zc.notna()
            num.loc[mask] += weight * zc.loc[mask]
            den.loc[mask] += abs(weight)
        data[out_col] = np.where(den > 0, num / den, np.nan)
        return data

    g = df.groupby("instrument", sort=False)
    eps = 1e-12

    def roll_shock(col, win=20, minp=5):
        m = g[col].transform(lambda s: s.shift(1).rolling(win, min_periods=minp).mean())
        sd = g[col].transform(lambda s: s.shift(1).rolling(win, min_periods=minp).std(ddof=0))
        return (df[col] - m) / (sd + eps)

    def roll_mean(col, win=5, minp=2):
        return g[col].transform(lambda s: s.rolling(win, min_periods=minp).mean())

    df["close_to_vwap"] = df["close"] / df["vwap"].replace(0, np.nan) - 1.0
    df["noise_ratio"] = (1.0 - df["path_efficiency"]).clip(lower=0.0, upper=1.0)
    df["impact_amihud"] = df["ret_cc"].abs() / (df["amount"] / 1e8 + 1.0)
    df["spread_shock_20d"] = roll_shock("spread_bps_mean")
    df["depth_shock_20d"] = roll_shock("depth5_mean")
    df["impact_shock_20d"] = roll_shock("impact_amihud")
    df["amount_shock_20d"] = roll_shock("amount")
    df["rv_shock_20d"] = roll_shock("realized_var")
    df["downside_share_shock_20d"] = roll_shock("downside_share")
    df["staleness_shock_20d"] = roll_shock("quote_staleness")
    df["concentration_shock_20d"] = roll_shock("max_minute_amount_share")
    df["average_trade_size_shock_20d"] = roll_shock("average_trade_amount")
    df["late_amount_shock_20d"] = roll_shock("amount_share_late30")
    ofi_std = g["ofi_sum"].transform(lambda s: s.shift(1).rolling(20, min_periods=5).std(ddof=0))
    df["ofi_persistence_5d"] = roll_mean("ofi_sum", 5) / (ofi_std + eps)
    df["signed_flow_5d"] = roll_mean("signed_amount_imbalance", 5)
    df["ofi_future5_corr_5d"] = roll_mean("ofi_future5_corr", 5)
    df["sell_absorption_good"] = (-df["signed_amount_imbalance"]).clip(lower=0) * df["ret_oc"].clip(lower=0)
    df["buy_absorption_bad"] = df["signed_amount_imbalance"].clip(lower=0) * (-df["ret_oc"]).clip(lower=0)
    df["absorption_score"] = df["sell_absorption_good"] - df["buy_absorption_bad"]
    df["absorption_5d"] = roll_mean("absorption_score", 5)
    df["resilience_score"] = z(df, "spread_recovery") + z(df, "depth_recovery") + z(df, "price_recovery")
    df["resilience_5d"] = roll_mean("resilience_score", 5)
    df["crowding_score"] = (
        z(df, "amount_shock_20d") + z(df, "max_minute_amount_share")
        + z(df, "amount_burstiness") + z(df, "average_trade_size_shock_20d")
    )
    df["downside_pressure"] = (
        z(df, "downside_share") + z(df, "rv_shock_20d")
        - z(df, "signed_amount_imbalance") + z(df, "spread_shock_20d")
    )
    df["late_orderbook_pressure"] = z(df, "obi5_tail") + z(df, "micro_dev1_tail")
    crowd_pos = (df["crowding_score"] > 0).astype(float)
    df["crowded_price_reversal"] = -df["ret_oc"] * crowd_pos
    df["late_crowding_reversal"] = -df["ret_late30"] * df["amount_share_late30"] * crowd_pos
    df = df.sort_values(["date", "instrument"]).reset_index(drop=True)
    df = make_composite(df, "factor_raw", [
            ("crowded_price_reversal", 1.0),
            ("late_crowding_reversal", 0.8),
            ("max_minute_amount_share", -0.5),
            ("amount_absret_corr", -0.3),
            ("crowding_score", -0.4),
        ])
    df["factor"] = z(df, "factor_raw")

    out = df[["date", "instrument", "factor"]].copy()
    out = out[(out["date"] >= start_ts.normalize()) & (out["date"] <= end_ts.normalize())]
    out["factor"] = pd.to_numeric(out["factor"], errors="coerce").replace([np.inf, -np.inf], np.nan)
    med = out.groupby("date")["factor"].transform("median")
    out["factor"] = out["factor"].fillna(med).fillna(0.0)
    out = out.drop_duplicates(["date", "instrument"], keep="last")
    out = out.sort_values(["date", "instrument"]).reset_index(drop=True)
    return out

In [ ]:
if __name__ == "__main__":
    from bigmodule import M
    import dai

    datasources = {
        "bar1m": "bigalpha_2026_stock_bar1m",
        "financial": "bigalpha_2026_financial",
        "instruments": "bigalpha_2026_instruments",
        "factorlib": "bigalpha_2026_factorlib",
    }

    start_date = "2019-01-01 00:00:00"
    end_date = "2024-12-31 23:59:59"

    factor_data = main(datasources, start_date, end_date)

    print(factor_data.head())
    print(factor_data["factor"].describe())
    print("zero_ratio:", (factor_data["factor"] == 0).mean())
    print("daily_count:")
    print(factor_data.groupby("date")["instrument"].count().describe())

    factor_pool = dai.query(
        "SELECT * FROM bigalpha_2026_factorlib",
        filters={"date": [start_date, end_date]},
        compression=True,
    ).df()

    result = M.bigalpha_eval._latest(
        factor_data=factor_data,
        factor_pool=factor_pool,
        process_pools=False,
        show=True,
    )